# Background

`1` Due to their low frequency, estimating the effects of rare variants is challenging.

- 특정 변이 $k$에 대한 개인 $i$의 Genotype을 $G \in \left \{0, 1, 2 \right \}$

<details>
<summary> MAF 문제</summary>

$$y_i = \mathbf{X}_i \boldsymbol{\alpha} + \mathbf{G}_i \boldsymbol{\beta} + \epsilon_i$$

$$\text{MAC}_k \text{ (Minor Allele Count)} = \sum_{i=1}^N G_{ik}$$

$$\text{MAF}_k = \frac{\text{MAC}_k}{2N} = \frac{\sum_{i=1}^N G_{ik}}{2N} \leq 0.01$$

$$\text{SE}(\hat{\beta}_k) = \frac{\sigma_\epsilon}{\sqrt{2N \cdot \text{MAF}_k(1 - \text{MAF}_k)}} \approx \infty$$

$$T_k = \frac{\hat{\beta}_k}{\text{SE}(\hat{\beta}_k)} \to 0$$

`2` To identify associations between rare variants and complex traits, gene-based or region-based tests$^{10,11}$, including the burden test, the sequence kernel association test (SKAT)$^{12}$ and its adaptive optimized version (SKAT-O)$^{13}$, have been commonly used.

`3` To address these challenges, we introduce RareEffect, an empirical Bayes approach designed to estimate the effect sizes of individual rare variants while simultaneously estimating gene-level heritability.

---

# Step 0. Full model 설정

`1` 포화모형

$$g(\boldsymbol{\mu}) = \mathbf{X}\boldsymbol{\alpha} + \mathbf{G}_j \boldsymbol{\beta}_j + \mathbf{b}, \quad \mathbf{b} \sim \text{MVN}(\mathbf{0}, \, \sigma_g^2 \mathbf{K})$$

# Step 1. Null model fiting

`1` $H_0: \beta_j=0$ 상태의 축소 모형을 적합

$$y = \mathbf{X}_i \boldsymbol{\alpha} + b_i + \varepsilon_i$$

$$\mathbf{b} \sim \text{MVN}(\mathbf{0}, \, \sigma_g^2 \mathbf{K}), \quad \boldsymbol{\varepsilon} \sim \text{MVN}(\mathbf{0}, \, \sigma_e^2 \mathbf{I}_n)$$

`2` We used the average information restricted maximum-likelihood algorithm to fit the null GLMM (that is, H0: βj = 0) as in SAIGE step 1.

`3` 도출 추정치

- $\hat {\alpha}$: 공변량 계수

- $\hat {\sigma_g}$: Polygenic Variance component

`4` 행렬 $\mathbf{K}$: Genentic Relatedenss Matrix, GRM 

- $\to$ 피험자들끼리 유전적으로 얼마나 닮았는지 나타냄.

    - $G_{im} \in \{0, 1, 2\}$: $i$번째 개인의 $m$번째 SNP 대립유전자 개수, $G_{im} \sim \text{Binomial}(2, p_m)$
 
    - $p_m$:  $m$번째 SNP의 대립유전자 빈도(MAF, Minor Allele Frequency)

<details>
    <summary>행렬 $\mathbf{K}$  세부 구성</summary>

$$\mathbf{K} = \frac{1}{M} \mathbf{Z}\mathbf{Z}^T$$

$$\mathbf{Z} = \begin{bmatrix} \frac{G_{11} - 2p_1}{\sqrt{2p_1(1-p_1)}} & \frac{G_{12} - 2p_2}{\sqrt{2p_2(1-p_2)}} & \cdots & \frac{G_{1M} - 2p_M}{\sqrt{2p_M(1-p_M)}} \\ \frac{G_{21} - 2p_1}{\sqrt{2p_1(1-p_1)}} & \frac{G_{22} - 2p_2}{\sqrt{2p_2(1-p_2)}} & \cdots & \frac{G_{2M} - 2p_M}{\sqrt{2p_M(1-p_M)}} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{G_{n1} - 2p_1}{\sqrt{2p_1(1-p_1)}} & \frac{G_{n2} - 2p_2}{\sqrt{2p_2(1-p_2)}} & \cdots & \frac{G_{nM} - 2p_M}{\sqrt{2p_M(1-p_M)}} \end{bmatrix}_{n \times M}$$

$$\begin{bmatrix} b_1 \\ b_2 \\ b_3 \\ \vdots \\ b_n \end{bmatrix} \sim \text{MVN}\left( \begin{bmatrix} 0 \\ 0 \\ 0 \\ \vdots \\ 0 \end{bmatrix}, \;\; \sigma_g^2 \begin{bmatrix} K_{11} & K_{12} & K_{13} & \cdots & K_{1n} \\ K_{21} & K_{22} & K_{23} & \cdots & K_{2n} \\ K_{31} & K_{32} & K_{33} & \cdots & K_{3n} \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ K_{n1} & K_{n2} & K_{n3} & \cdots & K_{nn} \end{bmatrix} \right)$$

</details>

`5` 그래서 최종 $\hat y$ 산출

$$\hat{y}_i = \hat{\mu}_i = X_i\hat{\boldsymbol{\alpha}} + \hat{b}_i$$

---

# Step 2. Estimation of gene-level heritability

## (1) 모형 설정

`1` 앞선 step1에서 공변량($X_i\hat{\boldsymbol{\alpha}}$)효과와 $\hat{b}_i$ 유전적 개인차 효과를 덜어낸,

`2` 잔차 벡터 $\tilde{\mathbf{y}}$를 반응변수로 두고, 특정 유전자 $j$의 단일 그룹에 대해 다음 모형을 세움.

$$\tilde{\mathbf{y}} = \mathbf{G}\boldsymbol{\beta} + \boldsymbol{\epsilon}, \quad \boldsymbol{\beta} \sim \text{MVN}(\mathbf{0}, \, \tau \boldsymbol{\Sigma}),\,\, \boldsymbol{\epsilon} \sim \text{MVN}(\mathbf{0}, \, \psi \mathbf{I}_n)$$

</details>

## (2) 변이를 그룹별로 쪼갠다?

`1` LoF (Loss-of-Function, 기능 상실 변이)

- 단백질